# Data Aggregation and Group Operations

In [2]:
import numpy as np
import pandas as pd
PREVIOUS_MAX_ROWS = pd.options.display.max_rows
pd.options.display.max_rows = 20
np.random.seed(12345)
import matplotlib.pyplot as plt
plt.rc('figure', figsize=(10, 6))
np.set_printoptions(precision=4, suppress=True)

In this
chapter, you will learn how to:
1. Split a pandas object into pieces using one or more keys (in the form of functions,
arrays, or DataFrame column names)
2. Calculate group summary statistics, like count, mean, or standard deviation, or a
user-defined function
3. Apply within-group transformations or other manipulations, like normalization,
linear regression, rank, or subset selection
4. Compute pivot tables and cross-tabulations
5. Perform quantile analysis and other statistical group analyses

ถ้าเจอเนื้อหาที่ correspond bullet points i ให้ใส่ tag ว่า #learn_i

>Aggregation of time series data, a special use case of groupby, is
referred to as resampling in this book and will receive separate
treatment in Chapter 11.

## GroupBy Mechanics

![](chan_folder/split-apply-combine.png)

Each grouping key can take many forms, and the keys do not have to be all of the
same type:
1. A list or array of values that is the same length as the axis being grouped
2. A value indicating a column name in a DataFrame
3. A dict or Series giving a correspondence between the values on the axis being grouped and the group names
3. A function to be invoked on the axis index or the individual labels in the index


ถ้าตรงไหนใช้ key type: i ก็ให้ใส่ tag #key_i

In [42]:
df = pd.DataFrame({'key1' : ['a', 'a', 'b', 'b', 'a'],
                   'key2' : ['one', 'two', 'one', 'two', 'one'],
                   'data1' : np.random.randn(5),
                   'data2' : np.random.randn(5)})
df

,key1,key2,data1,data2
0,a,one,-0.204708,1.393406
1,a,two,0.478943,0.092908
2,b,one,-0.519439,0.281746
3,b,two,-0.555730,0.769023
4,a,one,1.965781,1.246435


In [4]:
#key_1
grouped = df['data1'].groupby(df['key1'])
grouped

In [5]:
grouped.mean()

key1
a    0.746672
b   -0.537585
Name: data1, dtype: float64

Later, I’ll explain more about what happens when you call `.mean()`. The important
thing here is that the data (a Series) has been aggregated according to the group key,
producing a new Series that is now indexed by the unique values in the `key1` column

The result index has the name 'key1' because the DataFrame column `df['key1']`
did(see below.)

In [11]:
#ref_index
df['key1']

0    a
1    a
2    b
3    b
4    a
Name: key1, dtype: object

In [6]:
#key_1
means = df['data1'].groupby([df['key1'], df['key2']]).mean()
means
# เนื่องจากว่า keys=[a, one]|[b, one]|[b, two] ปรากฎแค่ครั้งเดียว mean ก็เท่ากับค่าใน df ดั้งเดิม

key1  key2
a     one     0.880536
      two     0.478943
b     one    -0.519439
      two    -0.555730
Name: data1, dtype: float64

In [7]:
means.unstack()

key2,one,two
key1,,
a,0.880536,0.478943
b,-0.519439,-0.555730


In [8]:
states = np.array(['Ohio', 'California', 'California', 'Ohio', 'Ohio'])
years = np.array([2005, 2005, 2006, 2005, 2006])
df['data1'].groupby([states, years]).mean()

California  2005    0.478943
            2006   -0.519439
Ohio        2005   -0.380219
            2006    1.965781
Name: data1, dtype: float64

In [9]:
#key_2
df.groupby('key1').mean()

,data1,data2
key1,,
a,0.746672,0.910916
b,-0.537585,0.525384


In [10]:
#key_2
df.groupby(['key1', 'key2']).mean()

data1     data2
key1 key2                    
a    one   0.880536  1.319920
     two   0.478943  0.092908
b    one  -0.519439  0.281746
     two  -0.555730  0.769023

You may have noticed in the first case `df.groupby('key1').mean()` that there is no
`key2` column in the result. Because `df['key2']` is not numeric data, it is said to be a
nuisance column, which is therefore excluded from the result. By default, all of the
numeric columns are aggregated, **though it is possible to filter down to a subset, as
you’ll see soon.**

ตรงที่ bold หมายถึง สามารถเลือกแค่บาง numeric columns เท่านั้นที่จะ ทำการ aggregate

In [12]:
df.groupby(['key1', 'key2']).size()

key1  key2
a     one     2
      two     1
b     one     1
      two     1
dtype: int64

### Iterating Over Groups

The GroupBy object supports iteration, generating a sequence of 2-tuples containing
the group name along with the chunk of data.

In [13]:
for name, group in df.groupby('key1'):
    print(name)
    print(group)

a
  key1 key2     data1     data2
0    a  one -0.204708  1.393406
1    a  two  0.478943  0.092908
4    a  one  1.965781  1.246435
b
  key1 key2     data1     data2
2    b  one -0.519439  0.281746
3    b  two -0.555730  0.769023


In [14]:
for (k1, k2), group in df.groupby(['key1', 'key2']):
    print((k1, k2))
    print(group)

('a', 'one')
  key1 key2     data1     data2
0    a  one -0.204708  1.393406
4    a  one  1.965781  1.246435
('a', 'two')
  key1 key2     data1     data2
1    a  two  0.478943  0.092908
('b', 'one')
  key1 key2     data1     data2
2    b  one -0.519439  0.281746
('b', 'two')
  key1 key2    data1     data2
3    b  two -0.55573  0.769023


In [15]:
# df.groupby(..) เป็น iterable ที่ element เป็น group name กับ data ถ้าแปลงเป็น list ก็จะเป็น list of 2-tuple ซึ่งสามารถ
# เอาไปสร้าง dict ได้ (เหมือนใน หมวด 3.1.4)
pieces = dict(list(df.groupby('key1')))
pieces['b']

,key1,key2,data1,data2
2,b,one,-0.519439,0.281746
3,b,two,-0.555730,0.769023


In [18]:
print(df.dtypes)
grouped = df.groupby(df.dtypes, axis=1)

key1      object
key2      object
data1    float64
data2    float64
dtype: object


In [22]:
for dtype, group in grouped:
    print(dtype)
    print(group)

float64
      data1     data2
0 -0.204708  1.393406
1  0.478943  0.092908
2 -0.519439  0.281746
3 -0.555730  0.769023
4  1.965781  1.246435
object
  key1 key2
0    a  one
1    a  two
2    b  one
3    b  two
4    a  one


#myexplain
ปกติ row ที่ ตำแหน่งที่ค่าใน column/Series ซ้ำกัน จะถูก grouped เข้าด้วยกัน โดย แต่ละ row จะเอาถูกเอามาทุกคอลัมน์

สำหรับกรณี axis=1 ดังข้างบน ตำแหน่ง 1,2 (หรือ 3,4) เป็นตำแหน่งที่ ค่าใน Series df.dtypes ซ้ำกัน ดังนั้นคอลัมน์ 1,2 (หรือ 3,4) ก็จะถูก grouped เข้าด้วยกัน โดยเวลาเอาคอลัมน์นึงมา จะเอามาทุกๆ row

### Selecting a Column or Subset of Columns

![](chan_folder/groupby_obj.png)

In [31]:
d = df.groupby('key1')
next(iter(d))
# d ไม่ใช่ dict และ 1st elemnt ของ tuple เป็น a กับ b ไม่มี data1

('a',
   key1 key2     data1     data2
 0    a  one -0.204708  1.393406
 1    a  two  0.478943  0.092908
 4    a  one  1.965781  1.246435)

```
df.groupby('key1')['data1']
df.groupby('key1')[['data2']]
```
เป้น syntactic sugar ของ
```
df['data1'].groupby(df['key1'])
df[['data2']].groupby(df['key1'])
```

In [32]:
df.groupby(['key1', 'key2'])[['data2']].mean()

data2
key1 key2          
a    one   1.319920
     two   0.092908
b    one   0.281746
     two   0.769023

#lower <br/>
The object returned by this indexing operation is a grouped DataFrame if a list or
array is passed or a grouped Series if only a single column name is passed as a scalar:

In [33]:
s_grouped = df.groupby(['key1', 'key2'])['data2']
s_grouped
s_grouped.mean()

key1  key2
a     one     1.319920
      two     0.092908
b     one     0.281746
      two     0.769023
Name: data2, dtype: float64

### Grouping with Dicts and Series

In [3]:
people = pd.DataFrame(np.random.randn(5, 5),
                      columns=['a', 'b', 'c', 'd', 'e'],
                      index=['Joe', 'Steve', 'Wes', 'Jim', 'Travis'])
people.iloc[2:3, [1, 2]] = np.nan # Add a few NA values
people

,a,b,c,d,e
Joe,-0.204708,0.478943,-0.519439,-0.555730,1.965781
Steve,1.393406,0.092908,0.281746,0.769023,1.246435
Wes,1.007189,NaN,NaN,0.228913,1.352917
Jim,0.886429,-2.001637,-0.371843,1.669025,-0.438570
Travis,-0.539741,0.476985,3.248944,-1.021228,-0.577087


In [4]:
mapping = {'a': 'red', 'b': 'red', 'c': 'blue',
           'd': 'blue', 'e': 'red', 'f' : 'orange'}
# column 'a' corresponds to red, 'b' to red, 'c' to blue, ...

In [5]:
by_column = people.groupby(mapping, axis=1)
by_column.sum()

,blue,red
Joe,-1.075169,2.240016
Steve,1.050769,2.732748
Wes,0.228913,2.360106
Jim,1.297183,-1.553778
Travis,2.227716,-0.639844


In [6]:
#retained
map_series = pd.Series(mapping)
map_series
people.groupby(map_series, axis=1).count()
# เนื่องจากเรา group columns, count is perform across column, และ rows are retained ทำให้ จน. row เท่ากับใน orig. data frame
# Because of missing values in row 'Wes'

,blue,red
Joe,2,3
Steve,2,3
Wes,1,2
Jim,2,3
Travis,2,3


In [85]:
#ลองเอง
for color, group in people.groupby(map_series, axis=1):
    print(color)
    print(group.join(group.count(axis=1).rename(index='count')), '\n')

blue
               c         d  count
Joe    -0.519439 -0.555730      2
Steve   0.281746  0.769023      2
Wes          NaN  0.228913      1
Jim    -0.371843  1.669025      2
Travis  3.248944 -1.021228      2 

red
               a         b         e  count
Joe    -0.204708  0.478943  1.965781      3
Steve   1.393406  0.092908  1.246435      3
Wes     1.007189       NaN  1.352917      2
Jim     0.886429 -2.001637 -0.438570      3
Travis -0.539741  0.476985 -0.577087      3 



### Grouping with Functions

Using Python functions is a more generic way of defining a group mapping compared
with a dict or Series. **Any function passed as a group key will be called once per index
value, with the return values being used as the group names**. More concretely, consider
the example DataFrame from the previous section, which has people’s first
names as index values. Suppose you wanted to group by the length of the names;
while you could compute an array of string lengths, it’s simpler to just pass the `len`
function:

In [19]:
people.groupby(len).sum()

,a,b,c,d,e
3,1.688911,-1.522694,-0.891281,1.342208,2.880128
5,1.393406,0.092908,0.281746,0.769023,1.246435
6,-0.539741,0.476985,3.248944,-1.021228,-0.577087


In [33]:
#ลองเอง # same result
ser = people.index.map(len).to_series()
ser.index = people.index
print(ser,'\n')

people.groupby(ser).sum()

Joe       3
Steve     5
Wes       3
Jim       3
Travis    6
dtype: int64 



,a,b,c,d,e
3,1.688911,-1.522694,-0.891281,1.342208,2.880128
5,1.393406,0.092908,0.281746,0.769023,1.246435
6,-0.539741,0.476985,3.248944,-1.021228,-0.577087


Mixing functions with arrays, dicts, or Series is not a problem as everything gets converted
to arrays internally:

In [34]:
key_list = ['one', 'one', 'one', 'two', 'two']
people.groupby([len, key_list]).min()

a         b         c         d         e
3 one -0.204708  0.478943 -0.519439 -0.555730  1.352917
  two  0.886429 -2.001637 -0.371843  1.669025 -0.438570
5 one  1.393406  0.092908  0.281746  0.769023  1.246435
6 two -0.539741  0.476985  3.248944 -1.021228 -0.577087

### Grouping by Index Levels

In [35]:
columns = pd.MultiIndex.from_arrays([['US', 'US', 'US', 'JP', 'JP'],
                                    [1, 3, 5, 1, 3]],
                                    names=['cty', 'tenor'])
hier_df = pd.DataFrame(np.random.randn(4, 5), columns=columns)
hier_df

cty          US                            JP          
tenor         1         3         5         1         3
0      0.124121  0.302614  0.523772  0.000940  1.343810
1     -0.713544 -0.831154 -2.370232 -1.860761 -0.860757
2      0.560145 -1.265934  0.119827 -1.063512  0.332883
3     -2.359419 -0.199543 -1.541996 -0.970736 -1.307030

In [36]:
#retained
hier_df.groupby(level='cty', axis=1).count()
# computation across columns, rows are retained
# คำตอบควรมี 4 แถว สองคอลัมน์(US, JP)

cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


In [37]:
#ลองเอง #bigpicture
hier_df.groupby(level='cty', axis=1).count().columns.name
## ในกรณีนี้ (axis=1)
# ค่าของ Column-index ซ้ำๆกันได้ กลายมาเป็น column-index ทีมี unique value
# ชื่อของ column-index กลายมาเป็น ชื่อของ column-index (ถ้าสำหรับข้างบน ชื่อของ column กลายมาเป็นชื่อของ index)
## ในกรณีที่ผ่านมาด้านบนๆ (axis=0)
# column ที่ใช้เป็น key ที่มีค่าซ้ำๆ กัน กลายมาเป็น row-index ที่มี unique value

'cty'

## Data Aggregation

While `quantile` is not explicitly implemented for GroupBy, it is a Series method and
thus available for use

In [44]:
print(df)
grouped = df.groupby('key1')
grouped['data1'].quantile(0.9)
# ที่ออกมาเป็น series เพราะ subset ด้วย single value โดยไม่มี bracket ซ้อนอีกชั้น

  key1 key2     data1     data2
0    a  one -0.204708  1.393406
1    a  two  0.478943  0.092908
2    b  one -0.519439  0.281746
3    b  two -0.555730  0.769023
4    a  one  1.965781  1.246435


key1
a    1.668413
b   -0.523068
Name: data1, dtype: float64

In [45]:
#ลองเอง 
# columns are retained ทำทีละกรุป กรุปที่ key1 == 'a' ทำทุกๆคอลัมน์ ต่อมา กรุปที่ key1 == 'b' ทำทุกๆคอลัมน์
grouped.quantile(0.9)

,data1,data2
key1,,
a,1.668413,1.364012
b,-0.523068,0.720295


In [46]:
def peak_to_peak(arr):
    return arr.max() - arr.min()
grouped.agg(peak_to_peak)

,data1,data2
key1,,
a,2.170488,1.300498
b,0.036292,0.487276


In [58]:
#crosscheck
print(df, '\n')
# for key1=='a'
df_a = df.loc[[0, 1, 4], ['data1', 'data2']].apply(peak_to_peak)
# for key1=='b'
df_b = df.loc[[2, 3], ['data1', 'data2']].apply(peak_to_peak)

pd.DataFrame([df_a, df_b])

  key1 key2     data1     data2
0    a  one -0.204708  1.393406
1    a  two  0.478943  0.092908
2    b  one -0.519439  0.281746
3    b  two -0.555730  0.769023
4    a  one  1.965781  1.246435 



,data1,data2
0,2.170488,1.300498
1,0.036292,0.487276


In [ ]:
grouped.describe()

### Column-Wise and Multiple Function Application

In [62]:
tips = pd.read_csv('examples/tips.csv')
# Add tip percentage of total bill
tips['tip_pct'] = tips['tip'] / tips['total_bill']
tips[:6]

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
5,25.29,4.71,No,Sun,Dinner,4,0.186240


In [14]:
grouped = tips.groupby(['day', 'smoker'])

**กรณีแรก** Single Series ถูก grouped

In [64]:
grouped_pct = grouped['tip_pct']
grouped_pct.agg('mean')

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

In [17]:
#ลองเอง #func_name 
# สังเกตว่่าผลลัพธ์จากบรรทัดที่สอง จะมีชื่อฟังชันก์มาเป็น column labels ด้วย
print(grouped.agg('mean'))
print(grouped.agg(['mean']))

             total_bill       tip      size   tip_pct
day  smoker                                          
Fri  No       18.420000  2.812500  2.250000  0.151650
     Yes      16.813333  2.714000  2.066667  0.174783
Sat  No       19.661778  3.102889  2.555556  0.158048
     Yes      21.276667  2.875476  2.476190  0.147906
Sun  No       20.506667  3.167895  2.929825  0.160113
     Yes      24.120000  3.516842  2.578947  0.187250
Thur No       17.113111  2.673778  2.488889  0.160298
     Yes      19.190588  3.030000  2.352941  0.163863
            total_bill       tip      size   tip_pct
                  mean      mean      mean      mean
day  smoker                                         
Fri  No      18.420000  2.812500  2.250000  0.151650
     Yes     16.813333  2.714000  2.066667  0.174783
Sat  No      19.661778  3.102889  2.555556  0.158048
     Yes     21.276667  2.875476  2.476190  0.147906
Sun  No      20.506667  3.167895  2.929825  0.160113
     Yes     24.120000  3.516842  2.

หนึ่งฟังชันก์ correspond to หนึ่งคอลัมน์ใน result
*หมายเหตุ* columns are retained ซึ่งเป็นจริงเพราะว่า Series เสมือนมี column เดียวอยู่แล้ว 
พอออกมาเป็น mean(หรือ std, peak_to_peak) แค่คอลัมน์เดียวก็ครบ

In [73]:
grouped_pct.agg(['mean', np.std, peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

สามารถตั้งชื่อ คอลัมน์ฺใน result ได้

In [66]:
grouped_pct.agg([('foo', 'mean'), ('bar', np.std)])

foo       bar
day  smoker                    
Fri  No      0.151650  0.028123
     Yes     0.174783  0.051293
Sat  No      0.158048  0.039767
     Yes     0.147906  0.061375
Sun  No      0.160113  0.042347
     Yes     0.187250  0.154134
Thur No      0.160298  0.038774
     Yes     0.163863  0.039389

**กรณ๊สอง** DataFrame ถูก grouped

In [69]:
#retained
functions = ['count', 'mean', 'max']
result = grouped[['tip_pct', 'total_bill']].agg(functions)
result
# columns are retained ในแง่ที่ว่า สำหรับทุกๆ func จะ operate บนทั้ง tip_pct และ total_bill

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

In [ ]:
result['tip_pct']

ตั้งชื่อให้ column ใน result

In [71]:
ftuples = [('Durchschnitt', 'mean'), ('Abweichung', np.var)]
grouped[['tip_pct', 'total_bill']].agg(ftuples)

tip_pct              total_bill            
            Durchschnitt Abweichung Durchschnitt  Abweichung
day  smoker                                                 
Fri  No         0.151650   0.000791    18.420000   25.596333
     Yes        0.174783   0.002631    16.813333   82.562438
Sat  No         0.158048   0.001581    19.661778   79.908965
     Yes        0.147906   0.003767    21.276667  101.387535
Sun  No         0.160113   0.001793    20.506667   66.099980
     Yes        0.187250   0.023757    24.120000  109.046044
Thur No         0.160298   0.001503    17.113111   59.625081
     Yes        0.163863   0.001551    19.190588   69.808518

key คือ ชื่อคอลัมน์ value คือชื่อ function

สำหรับข้างบน ทุกคอลัมน์ใน `['tip_pct', 'total_bill']` ถูก applied ด้วย ทุกฟังชันก์ `['count', 'mean', 'max']` (เหมือน cross product)

สำหรับข้างล่าง tip กับ size ถูก applied ด้วยแค่ฟังชันก์ที่อยู่ใน value เท่านั้น

In [74]:
grouped.agg({'tip' : np.max, 'size' : 'sum'})

tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [75]:
grouped.agg({'tip_pct' : ['min', 'max', 'mean', 'std'],
             'size' : 'sum'})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

>A DataFrame will have hierarchical columns only if multiple functions are applied to
at least one column.

### Returning Aggregated Data Without Row Indexes

In [76]:
tips.groupby(['day', 'smoker'], as_index=False).mean()

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


>Of course, it’s always possible to obtain the result in this format by calling
`reset_index` on the result. Using the `as_index=False` method avoids some unnecessary
computations.

## Apply: General split-apply-combine

In [77]:
def top(df, n=5, column='tip_pct'):
    return df.sort_values(by=column)[-n:]
top(tips, n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
232,11.61,3.39,No,Sat,Dinner,2,0.291990
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345


In [78]:
#with_group_keys
tips.groupby('smoker').apply(top)

total_bill   tip smoker   day    time  size   tip_pct
smoker                                                           
No     88        24.71  5.85     No  Thur   Lunch     2  0.236746
       185       20.69  5.00     No   Sun  Dinner     5  0.241663
       51        10.29  2.60     No   Sun  Dinner     2  0.252672
       149        7.51  2.00     No  Thur   Lunch     2  0.266312
       232       11.61  3.39     No   Sat  Dinner     2  0.291990
Yes    109       14.31  4.00    Yes   Sat  Dinner     2  0.279525
       183       23.17  6.50    Yes   Sun  Dinner     4  0.280535
       67         3.07  1.00    Yes   Sat  Dinner     1  0.325733
       178        9.60  4.00    Yes   Sun  Dinner     2  0.416667
       172        7.25  5.15    Yes   Sun  Dinner     2  0.710345

In [79]:
tips.groupby(['smoker', 'day']).apply(top, n=1, column='total_bill')

total_bill    tip smoker   day    time  size   tip_pct
smoker day                                                             
No     Fri  94        22.75   3.25     No   Fri  Dinner     2  0.142857
       Sat  212       48.33   9.00     No   Sat  Dinner     4  0.186220
       Sun  156       48.17   5.00     No   Sun  Dinner     6  0.103799
       Thur 142       41.19   5.00     No  Thur   Lunch     5  0.121389
Yes    Fri  95        40.17   4.73    Yes   Fri  Dinner     4  0.117750
       Sat  170       50.81  10.00    Yes   Sat  Dinner     3  0.196812
       Sun  182       45.35   3.50    Yes   Sun  Dinner     3  0.077178
       Thur 197       43.11   5.00    Yes  Thur   Lunch     4  0.115982

In [80]:
result = tips.groupby('smoker')['tip_pct'].describe()
result

,count,mean,std,min,25%,50%,75%,max
smoker,,,,,,,,
No,151.0,0.159328,0.039910,0.056797,0.136906,0.155625,0.185014,0.291990
Yes,93.0,0.163196,0.085119,0.035638,0.106771,0.153846,0.195059,0.710345


Recall from section 8.3
>When you unstack in a DataFrame, the level unstacked becomes the lowest level in
the result:

In [81]:
result.unstack('smoker')

       smoker
count  No        151.000000
       Yes        93.000000
mean   No          0.159328
       Yes         0.163196
std    No          0.039910
       Yes         0.085119
min    No          0.056797
       Yes         0.035638
25%    No          0.136906
       Yes         0.106771
50%    No          0.155625
       Yes         0.153846
75%    No          0.185014
       Yes         0.195059
max    No          0.291990
       Yes         0.710345
dtype: float64

Inside GroupBy, when you invoke a method like describe, it is actually just a shortcut
for:

```
f = lambda x: x.describe()
grouped.apply(f)
```

### Suppressing the Group Keys

In the preceding examples, you see that the resulting object has a hierarchical index
formed from the group keys along with the indexes of each piece of the original
object. You can disable this by passing `group_keys=False` to groupby:

In [82]:
tips.groupby('smoker', group_keys=False).apply(top)
# เปรียบเทียบกับ #with_group_keys

,total_bill,tip,smoker,day,time,size,tip_pct
88,24.71,5.85,No,Thur,Lunch,2,0.236746
185,20.69,5.00,No,Sun,Dinner,5,0.241663
51,10.29,2.60,No,Sun,Dinner,2,0.252672
149,7.51,2.00,No,Thur,Lunch,2,0.266312
232,11.61,3.39,No,Sat,Dinner,2,0.291990
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345


### Quantile and Bucket Analysis

In [83]:
frame = pd.DataFrame({'data1': np.random.randn(1000),
                      'data2': np.random.randn(1000)})
quartiles = pd.cut(frame.data1, 4)
quartiles[:10]

0     (0.489, 2.208]
1    (-2.956, -1.23]
2     (-1.23, 0.489]
3     (-1.23, 0.489]
4     (0.489, 2.208]
5     (0.489, 2.208]
6    (-2.956, -1.23]
7     (-1.23, 0.489]
8     (0.489, 2.208]
9     (-1.23, 0.489]
Name: data1, dtype: category
Categories (4, interval[float64]): [(-2.956, -1.23] < (-1.23, 0.489] < (0.489, 2.208] < (2.208, 3.928]]

In [86]:
def get_stats(group):
    return {'min': group.min(), 'max': group.max(),
            'count': group.count(), 'mean': group.mean()}
grouped = frame.data2.groupby(quartiles)
grouped.apply(get_stats)

data1                 
(-2.956, -1.23]  min       -3.399312
                 max        1.670835
                 count     96.000000
                 mean      -0.063122
(-1.23, 0.489]   min       -2.989741
                 max        3.260383
                 count    596.000000
                 mean      -0.003908
(0.489, 2.208]   min       -3.745356
                 max        2.954439
                 count    297.000000
                 mean       0.092315
(2.208, 3.928]   min       -1.929776
                 max        1.765640
                 count     11.000000
                 mean       0.030607
Name: data2, dtype: float64

In [90]:
#ลองเอง #same result #ref_cut #map_same_as_series
def get_stats_series(group):
    return pd.Series({'min': group.min(), 'max': group.max(),
            'count': group.count(), 'mean': group.mean()})
grouped = frame.data2.groupby(quartiles)
grouped.apply(get_stats_series)

data1                 
(-2.956, -1.23]  min       -3.399312
                 max        1.670835
                 count     96.000000
                 mean      -0.063122
(-1.23, 0.489]   min       -2.989741
                 max        3.260383
                 count    596.000000
                 mean      -0.003908
(0.489, 2.208]   min       -3.745356
                 max        2.954439
                 count    297.000000
                 mean       0.092315
(2.208, 3.928]   min       -1.929776
                 max        1.765640
                 count     11.000000
                 mean       0.030607
Name: data2, dtype: float64

In [98]:
#ลองเอง #transpose
def get_stats_transpose(group):
    return pd.Series({'min': group.min(), 'max': group.max(),
            'count': group.count(), 'mean': group.mean()}).to_frame().T
grouped = frame.data2.groupby(quartiles)
grouped.apply(get_stats_transpose)

,,min,max,count,mean
data1,,,,,
"(-2.956, -1.23]",0,-3.399312,1.670835,96.0,-0.063122
"(-1.23, 0.489]",0,-2.989741,3.260383,596.0,-0.003908
"(0.489, 2.208]",0,-3.745356,2.954439,297.0,0.092315
"(2.208, 3.928]",0,-1.929776,1.765640,11.0,0.030607


In [92]:
grouped.apply(get_stats).unstack()

,min,max,count,mean
data1,,,,
"(-2.956, -1.23]",-3.399312,1.670835,96.0,-0.063122
"(-1.23, 0.489]",-2.989741,3.260383,596.0,-0.003908
"(0.489, 2.208]",-3.745356,2.954439,297.0,0.092315
"(2.208, 3.928]",-1.929776,1.765640,11.0,0.030607


These were equal-length buckets; to compute equal-size buckets based on sample
quantiles, use qcut. I’ll pass `labels=False` to just get quantile numbers:

In [93]:
# Return quantile numbers
grouping = pd.qcut(frame.data1, 10, labels=False)
grouped = frame.data2.groupby(grouping)
grouped.apply(get_stats).unstack()

,min,max,count,mean
data1,,,,
0,-3.399312,1.670835,100.0,-0.086107
1,-1.801179,2.628441,100.0,0.051974
2,-2.925113,2.527939,100.0,-0.073978
3,-2.315555,3.260383,100.0,0.114297
4,-2.047939,2.074345,100.0,-0.127181
5,-2.989741,2.184810,100.0,0.021461
6,-2.084231,2.458842,100.0,-0.038834
7,-3.056990,2.954439,100.0,0.003504
8,-3.745356,2.735527,100.0,0.129800


In [94]:
#ลองเอง
grouping

0      8
1      0
2      6
3      5
4      9
      ..
995    6
996    6
997    4
998    9
999    5
Name: data1, Length: 1000, dtype: int64

In [95]:
#ลองเอง #name
# เทียบจาก #ref_cut --> data1 เปลี่ยนจาก interval เป็น number
# ที่ result index เป็น data1 เพราะ series ที่ใช้คือ grouping ซึ่งมี Name: data1 --> เหมือนที่อธิบายใน #ref_index
grouped.apply(get_stats)

data1       
0      min       -3.399312
       max        1.670835
       count    100.000000
       mean      -0.086107
1      min       -1.801179
                   ...    
8      mean       0.129800
9      min       -2.064111
       max        2.377020
       count    100.000000
       mean       0.198718
Name: data2, Length: 40, dtype: float64

In [97]:
#ลองเอง # ถ้าไม่ suppress labels
pd.qcut(frame.data1, 10)

0                     (0.822, 1.297]
1      (-2.9499999999999997, -1.212]
2                     (0.235, 0.503]
3                   (-0.0144, 0.235]
4                     (1.297, 3.928]
                   ...              
995                   (0.235, 0.503]
996                   (0.235, 0.503]
997                (-0.303, -0.0144]
998                   (1.297, 3.928]
999                 (-0.0144, 0.235]
Name: data1, Length: 1000, dtype: category
Categories (10, interval[float64]): [(-2.9499999999999997, -1.212] < (-1.212, -0.872] < (-0.872, -0.563] < (-0.563, -0.303] ... (0.235, 0.503] < (0.503, 0.822] < (0.822, 1.297] < (1.297, 3.928]]

### Example: Filling Missing Values with Group-Specific       Values

In [ ]:
s = pd.Series(np.random.randn(6))
s[::2] = np.nan
s
s.fillna(s.mean())

In [31]:
np.random.seed(123)
states = ['Ohio', 'New York', 'Vermont', 'Florida',
          'Oregon', 'Nevada', 'California', 'Idaho']
group_key = ['East'] * 4 + ['West'] * 4
data = pd.Series(np.random.randn(8), index=states)
data

Ohio         -1.086
New York      0.997
Vermont       0.283
Florida      -1.506
Oregon       -0.579
Nevada        1.651
California   -2.427
Idaho        -0.429
dtype: float64

In [33]:
data[['Vermont', 'Nevada', 'Idaho']] = np.nan
data
data.groupby(group_key).mean()

East   -0.532
West   -1.503
dtype: float64

In [34]:
fill_mean = lambda g: g.fillna(g.mean())
data.groupby(group_key).apply(fill_mean)
# Expect
# East: Vermont = -0.532
# West: Nevada, Idaho = -1.503

Ohio         -1.086
New York      0.997
Vermont      -0.532
Florida      -1.506
Oregon       -0.579
Nevada       -1.503
California   -2.427
Idaho        -1.503
dtype: float64

In [35]:
fill_values = {'East': 0.5, 'West': -1}
fill_func = lambda g: g.fillna(fill_values[g.name]) # g.name เป็น name ของ group
data.groupby(group_key).apply(fill_func)

Ohio         -1.086
New York      0.997
Vermont       0.500
Florida      -1.506
Oregon       -0.579
Nevada       -1.000
California   -2.427
Idaho        -1.000
dtype: float64

![](chan_folder/group_key.png)

### Example: Random Sampling and Permutation

In [ ]:
# Hearts, Spades, Clubs, Diamonds
suits = ['H', 'S', 'C', 'D']
card_val = (list(range(1, 11)) + [10] * 3) * 4
base_names = ['A'] + list(range(2, 11)) + ['J', 'K', 'Q']
cards = []
for suit in ['H', 'S', 'C', 'D']:
    cards.extend(str(num) + suit for num in base_names)

deck = pd.Series(card_val, index=cards)

In [ ]:
deck[:13]

In [ ]:
def draw(deck, n=5):
    return deck.sample(n)
draw(deck)

In [ ]:
get_suit = lambda card: card[-1] # last letter is suit
deck.groupby(get_suit).apply(draw, n=2)

In [ ]:
deck.groupby(get_suit, group_keys=False).apply(draw, n=2)

### Example: Group Weighted Average and Correlation

In [37]:
df = pd.DataFrame({'category': ['a', 'a', 'a', 'a',
                                'b', 'b', 'b', 'b'],
                   'data': np.random.randn(8),
                   'weights': np.random.rand(8)})
df

,category,data,weights
0,a,1.266,0.532
1,a,-0.867,0.532
2,a,-0.679,0.634
3,a,-0.095,0.849
4,b,1.491,0.724
5,b,-0.639,0.611
6,b,-0.444,0.722
7,b,-0.434,0.323


In [38]:
grouped = df.groupby('category')
get_wavg = lambda g: np.average(g['data'], weights=g['weights'])
grouped.apply(get_wavg)

category
a   -0.117
b    0.096
dtype: float64

In [39]:
close_px = pd.read_csv('examples/stock_px_2.csv', parse_dates=True,
                       index_col=0)
close_px.info()
close_px[-4:]

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2214 entries, 2003-01-02 to 2011-10-14
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2214 non-null   float64
 1   MSFT    2214 non-null   float64
 2   XOM     2214 non-null   float64
 3   SPX     2214 non-null   float64
dtypes: float64(4)
memory usage: 86.5 KB


,AAPL,MSFT,XOM,SPX
2011-10-11,400.29,27.00,76.27,1195.54
2011-10-12,402.19,26.96,77.16,1207.25
2011-10-13,408.43,27.18,76.37,1203.66
2011-10-14,422.00,27.27,78.11,1224.58


In [40]:
spx_corr = lambda x: x.corrwith(x['SPX'])

In [41]:
rets = close_px.pct_change().dropna() # dropna by default drops any row containing a missing value

In [57]:
#return_series_for_frame
def spx_corr_2(x):
    print(f"year={x.name}")
    print(x)
    # From Section 5.3
    # Passing a Series returns a Series with the correlation value computed for each column:
    c = x.corrwith(x['SPX'])
    print(c) # c is a Series. Consistent with the statement 2 lines above!
    return c

In [58]:
#date
get_year = lambda x: x.year
by_year = rets.groupby(get_year)
# Any function passed as a group key will be called once per index value, with the return values being used as the group names.

#by_year.apply(spx_corr)
by_year.apply(spx_corr_2)

year=2003
             AAPL   MSFT        XOM        SPX
2003-01-03  0.007  0.001  6.845e-04 -4.840e-04
2003-01-06  0.000  0.018  2.462e-02  2.247e-02
2003-01-07 -0.003  0.019 -3.371e-02 -6.545e-03
2003-01-08 -0.020 -0.028 -4.145e-03 -1.409e-02
2003-01-09  0.008  0.029  2.116e-02  1.939e-02
...           ...    ...        ...        ...
2003-12-24  0.030 -0.004  2.080e-03 -1.807e-03
2003-12-26  0.019  0.006  5.336e-03  1.691e-03
2003-12-29  0.017  0.009  1.327e-02  1.240e-02
2003-12-30  0.007  0.002  2.619e-03  1.442e-04
2003-12-31  0.005 -0.005  7.837e-03  2.055e-03

[251 rows x 4 columns]
AAPL    0.541
MSFT    0.745
XOM     0.661
SPX     1.000
dtype: float64
year=2004
             AAPL   MSFT    XOM        SPX
2004-01-02 -0.005  0.003 -0.009 -3.094e-03
2004-01-05  0.042  0.025  0.023  1.240e-02
2004-01-06 -0.004  0.004 -0.007  1.292e-03
2004-01-07  0.023 -0.001 -0.007  2.367e-03
2004-01-08  0.034 -0.001 -0.003  4.963e-03
...           ...    ...    ...        ...
2004-12-27 -0.013 -0

,AAPL,MSFT,XOM,SPX
2003,0.541,0.745,0.661,1.0
2004,0.374,0.589,0.558,1.0
2005,0.468,0.562,0.631,1.0
2006,0.428,0.406,0.519,1.0
2007,0.508,0.659,0.786,1.0
2008,0.681,0.805,0.828,1.0
2009,0.707,0.655,0.798,1.0
2010,0.710,0.730,0.839,1.0
2011,0.692,0.801,0.860,1.0


In [61]:
#ลองเอง # ลองทำให้ c เป็น DataFrame ดูว่า จะปรากฎ index เหมือนใน #with_group_keys มั้ย
def spx_corr_3(x):
    print(f"year={x.name}")
    print(x)
    # From Section 5.3
    # Passing a Series returns a Series with the correlation value computed for each column:
    c = x.corrwith(x['SPX'])
    c = pd.DataFrame([c])
    print(c)
    return c

by_year.apply(spx_corr_3)

year=2003
             AAPL   MSFT        XOM        SPX
2003-01-03  0.007  0.001  6.845e-04 -4.840e-04
2003-01-06  0.000  0.018  2.462e-02  2.247e-02
2003-01-07 -0.003  0.019 -3.371e-02 -6.545e-03
2003-01-08 -0.020 -0.028 -4.145e-03 -1.409e-02
2003-01-09  0.008  0.029  2.116e-02  1.939e-02
...           ...    ...        ...        ...
2003-12-24  0.030 -0.004  2.080e-03 -1.807e-03
2003-12-26  0.019  0.006  5.336e-03  1.691e-03
2003-12-29  0.017  0.009  1.327e-02  1.240e-02
2003-12-30  0.007  0.002  2.619e-03  1.442e-04
2003-12-31  0.005 -0.005  7.837e-03  2.055e-03

[251 rows x 4 columns]
    AAPL   MSFT    XOM  SPX
0  0.541  0.745  0.661  1.0
year=2004
             AAPL   MSFT    XOM        SPX
2004-01-02 -0.005  0.003 -0.009 -3.094e-03
2004-01-05  0.042  0.025  0.023  1.240e-02
2004-01-06 -0.004  0.004 -0.007  1.292e-03
2004-01-07  0.023 -0.001 -0.007  2.367e-03
2004-01-08  0.034 -0.001 -0.003  4.963e-03
...           ...    ...    ...        ...
2004-12-27 -0.013 -0.006 -0.021 -4.

,,AAPL,MSFT,XOM,SPX
2003,0,0.541,0.745,0.661,1.0
2004,0,0.374,0.589,0.558,1.0
2005,0,0.468,0.562,0.631,1.0
2006,0,0.428,0.406,0.519,1.0
2007,0,0.508,0.659,0.786,1.0
2008,0,0.681,0.805,0.828,1.0
2009,0,0.707,0.655,0.798,1.0
2010,0,0.710,0.730,0.839,1.0
2011,0,0.692,0.801,0.860,1.0


You could also compute inter-column correlations. Here we compute the annual correlation
between Apple and Microsoft:

In [62]:
by_year.apply(lambda g: g['AAPL'].corr(g['MSFT']))

2003    0.481
2004    0.259
2005    0.300
2006    0.162
2007    0.418
2008    0.612
2009    0.433
2010    0.572
2011    0.582
dtype: float64

### Example: Group-Wise Linear Regression

In the same theme as the previous example, you can use `groupby` to perform more
complex group-wise statistical analysis, **as long as the function returns a pandas
object or scalar value.** For example, I can define the following `regress` function
(using the `statsmodels` econometrics library), which executes an ordinary least
squares (OLS) regression on each chunk of data:

In [63]:
import statsmodels.api as sm
def regress(data, yvar, xvars):
    Y = data[yvar]
    X = data[xvars]
    X['intercept'] = 1.
    result = sm.OLS(Y, X).fit()
    return result.params

In [64]:
by_year.apply(regress, 'AAPL', ['SPX'])

,SPX,intercept
2003,1.195,7.100e-04
2004,1.363,4.201e-03
2005,1.766,3.246e-03
2006,1.645,7.957e-05
2007,1.199,3.438e-03
2008,0.968,-1.110e-03
2009,0.879,2.954e-03
2010,1.053,1.261e-03
2011,0.807,1.514e-03


## Pivot Tables and Cross-Tabulation

In [3]:
tips = pd.read_csv('examples/tips.csv')
# Add tip percentage of total bill
tips['tip_pct'] = tips['tip'] / tips['total_bill']
tips[:6]

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
5,25.29,4.71,No,Sun,Dinner,4,0.186240


Suppose you wanted to compute a table of group
means (the default pivot_table aggregation type) arranged by day and smoker on
the rows:

In [4]:
tips.pivot_table(index=['day', 'smoker'])

size       tip   tip_pct  total_bill
day  smoker                                          
Fri  No      2.250000  2.812500  0.151650   18.420000
     Yes     2.066667  2.714000  0.174783   16.813333
Sat  No      2.555556  3.102889  0.158048   19.661778
     Yes     2.476190  2.875476  0.147906   21.276667
Sun  No      2.929825  3.167895  0.160113   20.506667
     Yes     2.578947  3.516842  0.187250   24.120000
Thur No      2.488889  2.673778  0.160298   17.113111
     Yes     2.352941  3.030000  0.163863   19.190588

แปลย่อๆ ก็คือเพิ่ม group key มาแค่ time อย่างเดียว ส่วน key เดิม อันนึงไว้ ที่ row อีกอัน column

Now, suppose we want to aggregate only `tip_pct` and `size`, and additionally group by time. I’ll put `smoker` in
the table columns and `day` in the rows:

In [5]:
tips.pivot_table(['tip_pct', 'size'], index=['time', 'day'],
                 columns='smoker')

size             tip_pct          
smoker             No       Yes        No       Yes
time   day                                         
Dinner Fri   2.000000  2.222222  0.139622  0.165347
       Sat   2.555556  2.476190  0.158048  0.147906
       Sun   2.929825  2.578947  0.160113  0.187250
       Thur  2.000000       NaN  0.159744       NaN
Lunch  Fri   3.000000  1.833333  0.187735  0.188937
       Thur  2.500000  2.352941  0.160311  0.163863

In [19]:
#ลองเอง
tips.groupby(['time', 'day', 'smoker'])[['size', 'tip_pct']].agg('mean').unstack(level='smoker')

size             tip_pct          
smoker             No       Yes        No       Yes
time   day                                         
Dinner Fri   2.000000  2.222222  0.139622  0.165347
       Sat   2.555556  2.476190  0.158048  0.147906
       Sun   2.929825  2.578947  0.160113  0.187250
       Thur  2.000000       NaN  0.159744       NaN
Lunch  Fri   3.000000  1.833333  0.187735  0.188937
       Thur  2.500000  2.352941  0.160311  0.163863

In [20]:
tips.pivot_table(['tip_pct', 'size'], index=['time', 'day'],
                 columns='smoker', margins=True)

size                       tip_pct                    
smoker             No       Yes       All        No       Yes       All
time   day                                                             
Dinner Fri   2.000000  2.222222  2.166667  0.139622  0.165347  0.158916
       Sat   2.555556  2.476190  2.517241  0.158048  0.147906  0.153152
       Sun   2.929825  2.578947  2.842105  0.160113  0.187250  0.166897
       Thur  2.000000       NaN  2.000000  0.159744       NaN  0.159744
Lunch  Fri   3.000000  1.833333  2.000000  0.187735  0.188937  0.188765
       Thur  2.500000  2.352941  2.459016  0.160311  0.163863  0.161301
All          2.668874  2.408602  2.569672  0.159328  0.163196  0.160803

(คำอธิบายสำหรับข้างบน)
Here, the All values are means without taking into account smoker versus nonsmoker
(the All columns) or any of the two levels of grouping on the rows (the All
row).

To use a different aggregation function, pass it to aggfunc. For example, `'count'` or
`len` will give you a cross-tabulation (count or frequency) of group sizes:

In [26]:
tips.pivot_table('tip_pct', index=['time', 'smoker'], columns='day',
                 aggfunc=len, margins=True)

day             Fri   Sat   Sun  Thur    All
time   smoker                               
Dinner No       3.0  45.0  57.0   1.0  106.0
       Yes      9.0  42.0  19.0   NaN   70.0
Lunch  No       1.0   NaN   NaN  44.0   45.0
       Yes      6.0   NaN   NaN  17.0   23.0
All            19.0  87.0  76.0  62.0  244.0

In [31]:
# ลองใส่ bracket ดู เทียบกับ cell ก่อนหน้า
tips.pivot_table(['tip_pct'], index=['time', 'smoker'], columns='day',
                 aggfunc='count', margins=True)

tip_pct                       
day               Fri   Sat   Sun  Thur  All
time   smoker                               
Dinner No         3.0  45.0  57.0   1.0  106
       Yes        9.0  42.0  19.0   NaN   70
Lunch  No         1.0   NaN   NaN  44.0   45
       Yes        6.0   NaN   NaN  17.0   23
All              19.0  87.0  76.0  62.0  244

If some combinations are empty (or otherwise NA), you may wish to pass a `fill_value`:

In [32]:
tips.pivot_table('tip_pct', index=['time', 'size', 'smoker'],
                 columns='day', aggfunc='mean', fill_value=0)

day                      Fri       Sat       Sun      Thur
time   size smoker                                        
Dinner 1    No      0.000000  0.137931  0.000000  0.000000
            Yes     0.000000  0.325733  0.000000  0.000000
       2    No      0.139622  0.162705  0.168859  0.159744
            Yes     0.171297  0.148668  0.207893  0.000000
       3    No      0.000000  0.154661  0.152663  0.000000
...                      ...       ...       ...       ...
Lunch  3    Yes     0.000000  0.000000  0.000000  0.204952
       4    No      0.000000  0.000000  0.000000  0.138919
            Yes     0.000000  0.000000  0.000000  0.155410
       5    No      0.000000  0.000000  0.000000  0.121389
       6    No      0.000000  0.000000  0.000000  0.173706

[21 rows x 4 columns]

![](chan_folder/pivot_table.png)

### Cross-Tabulations: Crosstab

A cross-tabulation (or crosstab for short) is a special case of a pivot table that computes
group frequencies. Here is an example:

In [33]:
from io import StringIO
data = """\
Sample  Nationality  Handedness
1   USA  Right-handed
2   Japan    Left-handed
3   USA  Right-handed
4   Japan    Right-handed
5   Japan    Left-handed
6   Japan    Right-handed
7   USA  Right-handed
8   USA  Left-handed
9   Japan    Right-handed
10  USA  Right-handed"""
data = pd.read_table(StringIO(data), sep='\s+')

In [34]:
data

,Sample,Nationality,Handedness
0,1,USA,Right-handed
1,2,Japan,Left-handed
2,3,USA,Right-handed
3,4,Japan,Right-handed
4,5,Japan,Left-handed
5,6,Japan,Right-handed
6,7,USA,Right-handed
7,8,USA,Left-handed
8,9,Japan,Right-handed
9,10,USA,Right-handed


As part of some survey analysis, we might want to summarize this data by nationality
and handedness. You could use `pivot_table` to do this, but the `pandas.crosstab`
function can be more convenient:

In [35]:
pd.crosstab(data.Nationality, data.Handedness, margins=True)

Handedness,Left-handed,Right-handed,All
Nationality,,,
Japan,2,3,5
USA,1,4,5
All,3,7,10


The first two arguments to crosstab can each either be an array or Series or a list of
arrays. As in the `tips` data:

In [36]:
pd.crosstab([tips.time, tips.day], tips.smoker, margins=True)

smoker        No  Yes  All
time   day                
Dinner Fri     3    9   12
       Sat    45   42   87
       Sun    57   19   76
       Thur    1    0    1
Lunch  Fri     1    6    7
       Thur   44   17   61
All          151   93  244

In [65]:
#ลองเอง 
# same result

x = tips.groupby(['time', 'day', 'smoker'])['tip'].count().unstack()
x = x.fillna(0)
x = pd.concat([x, x.apply(np.sum, axis=1)], axis=1).rename(columns={0: 'All'})

y = x.apply(np.sum)
y = pd.DataFrame([y], index=[['All'], ['']])
x = pd.concat([x, y])

x

No   Yes    All
time   day                     
Dinner Fri     3.0   9.0   12.0
       Sat    45.0  42.0   87.0
       Sun    57.0  19.0   76.0
       Thur    1.0   0.0    1.0
Lunch  Fri     1.0   6.0    7.0
       Thur   44.0  17.0   61.0
All          151.0  93.0  244.0

In [ ]:
pd.options.display.max_rows = PREVIOUS_MAX_ROWS

## Conclusion